In [3]:
from pathlib import Path
import pandas as pd
from tqdm import tqdm
import os
GRAPH_FOLDER = "dataset/database_v3/Graph_Proj"

In [4]:
GRAPH_FOLDERS = [
    "dataset/database_v3/Graph_Proj",
    "dataset/database_v3/Graph_Size",
]

EDGE_COLUMN_MAP = {
    "Graph_Proj": ("project_id", "project_id", "neig_project_id"),
    "Graph_Size": ("node_id", "node_id", "neig_id"),
}


def count_ccr_graph(graph_folder):
    graph_path = Path(graph_folder)
    dataset = graph_path.name
    ccr_key_col, src_col, dst_col = EDGE_COLUMN_MAP[dataset]

    ccr_nodes = pd.read_csv(graph_path / "node_id_ccr.csv")
    ccr_node_set = set(ccr_nodes[ccr_key_col])

    edge_rows = []
    for edge_path in sorted((graph_path / "edges").glob("*.csv")):
        edges = pd.read_csv(edge_path)
        ccr_edges = edges[
            edges[src_col].isin(ccr_node_set) & edges[dst_col].isin(ccr_node_set)
        ]
        edge_rows.append({"edge_file": edge_path.name, "ccr_edges": len(ccr_edges)})

    edge_counts = pd.DataFrame(edge_rows)
    summary = {
        "dataset": dataset,
        "ccr_nodes": len(ccr_nodes),
        "ccr_edges": int(edge_counts["ccr_edges"].sum()),
    }
    return summary, edge_counts


summaries = []
edge_count_tables = {}
for graph_folder in GRAPH_FOLDERS:
    summary, edge_counts = count_ccr_graph(graph_folder)
    summaries.append(summary)
    edge_count_tables[summary["dataset"]] = edge_counts

summary_df = pd.DataFrame(summaries)
summary_df

      dataset  ccr_nodes  ccr_edges
0  Graph_Proj        803      35686
1  Graph_Size       2314      74098

In [5]:
detail_df = pd.concat(
    edge_counts.assign(dataset=dataset)
    for dataset, edge_counts in edge_count_tables.items()
).loc[:, ["dataset", "edge_file", "ccr_edges"]]
detail_df

       dataset                   edge_file  ccr_edges
0   Graph_Proj                dist_250.csv      10220
1   Graph_Proj             mrt_cir_500.csv      16148
2   Graph_Proj  mrt_nearest_dist_eps_1.csv       1824
3   Graph_Proj  mrt_nearest_dist_eps_2.csv       3518
4   Graph_Proj     same_condo_age_2026.csv       3976
5   Graph_Size                dist_250.csv      19140
6   Graph_Size             mrt_cir_500.csv      30994
7   Graph_Size  mrt_nearest_dist_eps_1.csv       3284
8   Graph_Size  mrt_nearest_dist_eps_2.csv       6294
9   Graph_Size     same_condo_age_2026.csv       8478
10  Graph_Size    same_project_node_id.csv       5908